# PFE ML Training Only

This notebook runs only the model-training step on an already-built data lake. It does not rebuild BODACC, clean tables, or feature tables.

Use this after the full pipeline/audits have already produced `data-lake/features/company_year_features` and `data-lake/features/risk_labels` in Google Drive.

## 1. Runtime

Use **High-RAM CPU**. GPU/TPU is not needed for the current logistic-regression baseline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

START_YEAR = 2017
TRAIN_END_YEAR = 2024
TRAIN_MAX_ROWS = 2_000_000
TARGET = 'continuity_risk_12m_label'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR    =', BACKEND_DIR)
print('DRIVE_ROOT     =', DRIVE_ROOT)
print('BRANCH         =', BRANCH)
print('TARGET         =', TARGET)
print('TRAIN_END_YEAR =', TRAIN_END_YEAR)
print('TRAIN_MAX_ROWS =', TRAIN_MAX_ROWS)


## 2. Pull Code And Install Training Dependencies

`matplotlib` is installed through `collabs/requirements-colab.txt` because Colab is the runtime that generates report images.

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!ls -lah collabs/requirements-colab.txt
!pip install -q -r collabs/requirements-colab.txt


## 3. Quick Dataset Check

This confirms that feature and label Parquet files exist before training starts.

In [ ]:
import duckdb

DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

con = duckdb.connect()
dataset_check = con.execute(f"""
    SELECT
      COUNT(*) AS joined_rows,
      MIN(f.prediction_year) AS min_year,
      MAX(f.prediction_year) AS max_year,
      SUM(l.{TARGET}::INTEGER) AS positive_rows,
      AVG(l.{TARGET}::INTEGER) AS positive_rate
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l
      USING (siren, prediction_year)
    WHERE l.{TARGET} IS NOT NULL
      AND f.prediction_year >= {START_YEAR}
      AND f.prediction_year <= {TRAIN_END_YEAR}
""").df()
display(dataset_check)

year_balance = con.execute(f"""
    SELECT f.prediction_year,
           COUNT(*) AS rows,
           SUM(l.{TARGET}::INTEGER) AS positive_rows,
           AVG(l.{TARGET}::INTEGER) AS positive_rate
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l
      USING (siren, prediction_year)
    WHERE l.{TARGET} IS NOT NULL
      AND f.prediction_year >= {START_YEAR}
      AND f.prediction_year <= {TRAIN_END_YEAR}
    GROUP BY f.prediction_year
    ORDER BY f.prediction_year
""").df()
display(year_balance)
con.close()


## 4. Train Baseline Model

The trainer creates a descriptive run folder under `$DRIVE_ROOT/ml-artifacts/runs/` and writes report-ready images, CSV tables, metadata, and `run_summary.md`.

In [ ]:
import json
import shlex
import subprocess
import sys

train_command = [
    sys.executable,
    '-u',
    '-m', 'app.tools.train_continuity_model',
    '--data-lake-dir', f'{DRIVE_ROOT}/data-lake',
    '--artifacts-dir', f'{DRIVE_ROOT}/ml-artifacts',
    '--target', TARGET,
    '--train-start-year', str(START_YEAR),
    '--train-end-year', str(TRAIN_END_YEAR),
    '--max-rows', str(TRAIN_MAX_ROWS),
    '--min-rows', '1000',
]

print(' '.join(shlex.quote(part) for part in train_command))
subprocess.run(train_command, check=True)

metadata_path = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_metadata.json'
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
print(json.dumps({
    'run_name': metadata.get('run_name'),
    'model_version': metadata.get('model_version'),
    'rows': metadata.get('rows'),
    'split_strategy': metadata.get('split_strategy'),
    'sample_strategy': metadata.get('sample_strategy'),
    'test_class_counts': metadata.get('test_class_counts'),
    'metrics': metadata.get('metrics'),
    'run_artifacts_dir': metadata.get('run_artifacts_dir'),
}, indent=2))


## 5. Display Report Artifacts

Use these files directly in the final report. The Markdown summary is the first thing to send for review after a training run.

In [ ]:
from IPython.display import Image, Markdown, display

run_dir = Path(metadata['run_artifacts_dir'])
print('Run folder:')
print(run_dir)

print('\nFiles:')
for path in sorted(run_dir.iterdir()):
    print(path.name)

summary_path = run_dir / 'run_summary.md'
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))

image_names = [
    'class_counts_by_year.png',
    'precision_recall_curve.png',
    'roc_curve.png',
    'confusion_matrix_at_0_5.png',
    'score_distribution_by_class.png',
    'threshold_tradeoff.png',
    'top_feature_coefficients.png',
]
for image_name in image_names:
    image_path = run_dir / image_name
    if image_path.exists():
        print('\n' + image_name)
        display(Image(filename=str(image_path)))

comparison_image = Path(DRIVE_ROOT) / 'ml-artifacts' / 'model_run_comparison.png'
if comparison_image.exists():
    print('\nmodel_run_comparison.png')
    display(Image(filename=str(comparison_image)))


## 6. What To Send After The Run

Send these files for review:

- `$DRIVE_ROOT/ml-artifacts/model_metadata.json`
- the latest run folder name under `$DRIVE_ROOT/ml-artifacts/runs/`
- `run_summary.md`
- `precision_recall_curve.png`
- `threshold_tradeoff.png`
- `model_run_comparison.png` if there is more than one run